# PyTorch Exercises: From Basics to Custom Backprop

**TIFR PreSchool 2026 — ML School**

This notebook is a graded set of exercises. Each section introduces a concept and then asks you to fill in the empty solution cell that follows.

**How to use this notebook**
- Run the *setup* cell first.
- Read the prompt, then write your answer in the cell marked `# Your code here`.
- Some exercises include `assert` statements — your solution is (probably) correct when they pass silently.
- If you get stuck, look up the relevant function in the [PyTorch docs](https://pytorch.org/docs/stable/index.html).

**Outline**

1. Tensors and basic operations
2. Broadcasting, indexing, reshaping
3. Autograd — automatic differentiation
4. Building models with `nn.Module`
5. Datasets, `DataLoader`, and a training loop
6. Custom loss functions
7. Custom optimizers (writing your own SGD, Momentum, RMSProp, Adam)
8. Custom autograd functions (forward + backward by hand)
9. Hooks, gradient surgery, higher-order gradients
10. Putting it all together: a tiny end-to-end project

## Setup

Run this cell once at the start of the session.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

torch.manual_seed(0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch version:', torch.__version__)
print('Device:', device)

---
## 1. Tensors and basic operations

A `torch.Tensor` is a multi-dimensional array with a `dtype`, a `device`, and (optionally) a `requires_grad` flag.

Useful constructors:
- `torch.tensor(data)` — from a Python list / NumPy array
- `torch.zeros`, `torch.ones`, `torch.eye`, `torch.arange`, `torch.linspace`
- `torch.randn`, `torch.rand`, `torch.randint`
- `.to(device)`, `.float()`, `.long()` for conversions

### Exercise 1.1 — Build a few tensors

Create the following tensors:

1. `a` — a `float32` tensor of shape `(3, 4)` filled with zeros.
2. `b` — the 5×5 identity matrix.
3. `c` — a tensor containing the integers 0, 2, 4, ..., 18 (i.e. even numbers from 0 up to 18 inclusive).
4. `d` — a `(2, 3, 4)` tensor of standard-normal random numbers.

Print each tensor's `shape` and `dtype`.

In [ ]:
# Exercise 1.1 — Build a few tensors

# 1. float32 zeros of shape (3, 4)
a = torch.zeros(3, 4, dtype=torch.float32)
print("a:", a.shape, a.dtype)

# 2. 5x5 identity matrix
b = torch.eye(5)
print("b:", b.shape, b.dtype)

# 3. Even integers 0, 2, 4, ..., 18
c = torch.arange(0, 19, 2)
print("c:", c.shape, c.dtype)

# 4. (2, 3, 4) standard-normal random tensor
d = torch.randn(2, 3, 4)
print("d:", d.shape, d.dtype)


### Exercise 1.2 — Element-wise math vs. matrix math

Given two random matrices `X` of shape `(4, 3)` and `Y` of shape `(3, 5)`:

1. Compute their **matrix product** `Z` of shape `(4, 5)` two different ways: with `@` and with `torch.matmul`.
2. Compute the element-wise product of `X` with itself.
3. Compute the sum of all elements of `Z`, the column-wise mean (one value per column), and the row-wise max.

Verify the two matrix-product results agree using `torch.allclose`.

In [ ]:
# Exercise 1.2 — Element-wise math vs. matrix math

X = torch.randn(4, 3)
Y = torch.randn(3, 5)

# Matrix product two ways
Z1 = X @ Y
Z2 = torch.matmul(X, Y)
print("Matrix products agree:", torch.allclose(Z1, Z2))

# Element-wise product of X with itself
X_sq = X * X
print("X*X shape:", X_sq.shape)

# Reductions on Z
print("Sum of all Z elements:", Z1.sum().item())
print("Column-wise mean:", Z1.mean(dim=0))
print("Row-wise max:", Z1.max(dim=1).values)


---
## 2. Broadcasting, indexing, reshaping

Broadcasting lets you operate on tensors of different shapes without writing loops, as long as their trailing dimensions are *compatible* (equal, or one of them is 1).

### Exercise 2.1 — Standardise the columns of a matrix

Given a matrix `M` of shape `(N, D)`, write code that produces `M_std` where each column has mean 0 and standard deviation 1. **Use broadcasting — no explicit loops.**

After your code, the asserts below should pass.

In [ ]:
M = torch.randn(100, 5) * torch.tensor([1., 2., 3., 4., 5.]) + torch.tensor([10., -3., 0., 7., 1.])

# Exercise 2.1 — Standardise columns using broadcasting
col_mean = M.mean(dim=0)          # shape (5,)
col_std  = M.std(dim=0, unbiased=False)  # shape (5,)
M_std = (M - col_mean) / col_std  # broadcast over rows

assert torch.allclose(M_std.mean(dim=0), torch.zeros(5), atol=1e-5)
assert torch.allclose(M_std.std(dim=0, unbiased=False), torch.ones(5), atol=1e-5)
print('OK')


### Exercise 2.2 — Reshape and permute

Start from `T = torch.arange(24)`.

1. Reshape `T` to shape `(2, 3, 4)` and call it `T1`.
2. Permute `T1`'s axes so the result has shape `(4, 2, 3)`. Call it `T2`.
3. Flatten `T2` back to a 1-D tensor `T3`.
4. Is `T3` equal to `T`? Why or why not?

In [ ]:
# Exercise 2.2 — Reshape and permute

T = torch.arange(24)

# 1. Reshape to (2, 3, 4)
T1 = T.reshape(2, 3, 4)
print("T1 shape:", T1.shape)

# 2. Permute axes: (2,3,4) -> (4,2,3)
T2 = T1.permute(2, 0, 1)
print("T2 shape:", T2.shape)

# 3. Flatten back to 1D
T3 = T2.flatten()
print("T3 shape:", T3.shape)

# 4. Is T3 == T?
print("T3 equals T:", torch.equal(T3, T))
# NO — permute reorders elements, so flattening a permuted tensor
# does not recover the original linear order. T is [0,1,...,23];
# T3 has a different element ordering because permute changed which
# axis is traversed first during flattening.


### Exercise 2.3 — Boolean / fancy indexing

Given `x = torch.randn(1000)`:

1. Build a mask that selects the entries of `x` strictly greater than 1.
2. Replace those entries (in a copy `y`) with the value `1.0`. The original `x` should be untouched.
3. Report the fraction of entries that were clipped.

In [ ]:
# Exercise 2.3 — Boolean / fancy indexing

x = torch.randn(1000)

# 1. Boolean mask: entries > 1
mask = x > 1.0

# 2. Copy x, clip masked entries to 1.0
y = x.clone()
y[mask] = 1.0

# Verify x is untouched
assert not torch.equal(x, y) or mask.sum() == 0

# 3. Fraction clipped
fraction = mask.float().mean().item()
print(f"Fraction clipped: {fraction:.3f}")
print(f"x max still: {x.max().item():.3f}  |  y max: {y.max().item():.3f}")


---
## 3. Autograd

PyTorch's `autograd` engine records operations on tensors that have `requires_grad=True` into a dynamic computation graph, then computes gradients on demand via `.backward()`.

Key ideas:
- Only **leaf** tensors with `requires_grad=True` accumulate gradients in `.grad`.
- `.backward()` traverses the graph in reverse, multiplying Jacobian-vector products.
- Call `tensor.detach()` to break a tensor out of the graph; wrap code in `torch.no_grad()` to disable tracking entirely.

### Exercise 3.1 — Gradient of a scalar function by hand and by autograd

Let $f(x) = 3x^3 - 5x^2 + 2x - 7$. Analytically, $f'(x) = 9x^2 - 10x + 2$.

1. Create `x = torch.tensor(2.0, requires_grad=True)`.
2. Compute `y = f(x)`, then `y.backward()`.
3. Check that `x.grad` equals $9 \cdot 2^2 - 10 \cdot 2 + 2 = 18$.

In [ ]:
# Exercise 3.1 — Gradient of a scalar function

x = torch.tensor(2.0, requires_grad=True)

# f(x) = 3x^3 - 5x^2 + 2x - 7
y = 3*x**3 - 5*x**2 + 2*x - 7
y.backward()

# f'(x) = 9x^2 - 10x + 2; at x=2: 9*4 - 20 + 2 = 18
print("x.grad:", x.grad.item())          # expect 18.0
print("Analytic:", 9*2**2 - 10*2 + 2)   # 18
assert torch.isclose(x.grad, torch.tensor(18.0))
print("OK")


### Exercise 3.2 — Gradient of a vector-valued function

Let $\mathbf{w} \in \mathbb{R}^3$ and $\mathbf{x} \in \mathbb{R}^3$, and define
$$L = \tfrac{1}{2} \| \mathbf{w} \odot \mathbf{x} - \mathbf{t} \|_2^2$$
for some target `t`.

Compute `dL/dw` for the given numerical values, using autograd. Then check it against the closed-form expression $\nabla_\mathbf{w} L = (\mathbf{w}\odot\mathbf{x} - \mathbf{t}) \odot \mathbf{x}$.

In [ ]:
w = torch.tensor([0.5, -1.0, 2.0], requires_grad=True)
x = torch.tensor([1.0, 2.0, -0.5])
t = torch.tensor([0.0, 0.0, 1.0])

# Exercise 3.2 — Gradient of vector-valued loss
L = 0.5 * ((w * x - t)**2).sum()
L.backward()

# Closed form: dL/dw = (w*x - t) * x
analytic = (w.detach() * x - t) * x
print("autograd grad:", w.grad)
print("analytic grad:", analytic)
assert torch.allclose(w.grad, analytic, atol=1e-6)
print("OK")


### Exercise 3.3 — `detach`, `no_grad`, and gradient accumulation

Build a small experiment that demonstrates each of these behaviours. For a parameter `p = torch.tensor(1.0, requires_grad=True)`:

1. Compute the gradient of `loss = p**2` twice in a row, **without** calling `p.grad.zero_()` in between. What is `p.grad` after the second call?
2. Repeat, but call `p.grad.zero_()` between the two backward passes.
3. Show that `(p.detach() ** 2).backward()` fails (or produces no gradient).
4. Show that operations inside `with torch.no_grad():` produce tensors with `requires_grad=False`.

In [ ]:
# Exercise 3.3 — detach, no_grad, gradient accumulation

p = torch.tensor(1.0, requires_grad=True)

# 1. Accumulation without zeroing
loss = p**2
loss.backward()
loss = p**2
loss.backward()
print("grad after 2 backward calls (no zero_):", p.grad)  # expect 4.0 (2+2)

# 2. With zeroing in between
p.grad.zero_()
loss = p**2
loss.backward()
p.grad.zero_()
loss = p**2
loss.backward()
print("grad after zero_ between calls:", p.grad)  # expect 2.0

# 3. detach breaks the graph — backward will fail
try:
    (p.detach()**2).backward()
    print("Should not reach here")
except RuntimeError as e:
    print("detach backward correctly failed:", str(e)[:60])

# 4. no_grad produces tensors with requires_grad=False
with torch.no_grad():
    q = p * 2
print("requires_grad inside no_grad:", q.requires_grad)  # False


---
## 4. Building models with `nn.Module`

Anything trainable in PyTorch is conventionally a subclass of `nn.Module`. You register `nn.Parameter`s (which automatically appear in `.parameters()`) and submodules; you implement `forward(...)`.

Building blocks worth knowing:
- `nn.Linear`, `nn.Conv2d`, `nn.LayerNorm`, `nn.Embedding`
- `nn.ReLU`, `nn.GELU`, `nn.Tanh`
- `nn.Sequential`, `nn.ModuleList`, `nn.ModuleDict`

### Exercise 4.1 — An MLP from scratch (no `nn.Linear`)

Implement a 2-layer MLP **without** using `nn.Linear` — instantiate the weights and biases yourself as `nn.Parameter`s. The model should map `D_in → H → D_out` with a `tanh` non-linearity in between.

Initialise weights with Kaiming-style scaling: `W ~ N(0, 2/fan_in)`.

In [ ]:
class MyMLP(nn.Module):
    def __init__(self, d_in, hidden, d_out):
        super().__init__()
        # Kaiming init: W ~ N(0, 2/fan_in)
        self.W1 = nn.Parameter(torch.randn(d_in, hidden) * (2.0 / d_in) ** 0.5)
        self.b1 = nn.Parameter(torch.zeros(hidden))
        self.W2 = nn.Parameter(torch.randn(hidden, d_out) * (2.0 / hidden) ** 0.5)
        self.b2 = nn.Parameter(torch.zeros(d_out))

    def forward(self, x):
        h = torch.tanh(x @ self.W1 + self.b1)
        return h @ self.W2 + self.b2

# Smoke test
m = MyMLP(8, 16, 4)
print("output shape:", m(torch.randn(5, 8)).shape)   # torch.Size([5, 4])
print("param count:", sum(p.numel() for p in m.parameters()))  # 8*16+16+16*4+4 = 212


### Exercise 4.2 — The same MLP using `nn.Sequential`

Re-implement the model above using `nn.Linear` layers wrapped in `nn.Sequential`. Confirm that the parameter count matches your hand-built version.

In [ ]:
# Exercise 4.2 — MLP via nn.Sequential

seq_mlp = nn.Sequential(
    nn.Linear(8, 16),
    nn.Tanh(),
    nn.Linear(16, 4)
)

# Kaiming init to match MyMLP
nn.init.kaiming_normal_(seq_mlp[0].weight, nonlinearity='tanh')
nn.init.kaiming_normal_(seq_mlp[2].weight, nonlinearity='tanh')

print("output shape:", seq_mlp(torch.randn(5, 8)).shape)
print("param count:", sum(p.numel() for p in seq_mlp.parameters()))  # expect 212


### Exercise 4.3 — Parameter counting and `.state_dict()`

Write a function `count_params(model, trainable_only=True)` that returns the total number of parameters. Use it on your MLP.

Then save the model's `state_dict()` to a file and load it back into a fresh instance. Verify the outputs match on a random input.

In [ ]:
# Exercise 4.3 — Parameter counting and state_dict

def count_params(model, trainable_only=True):
    total = 0
    for p in model.parameters():
        if trainable_only and not p.requires_grad:
            continue
        total += p.numel()
    return total

m = MyMLP(8, 16, 4)
print("Param count:", count_params(m))

# Save and reload state_dict
import tempfile, os
with tempfile.NamedTemporaryFile(suffix=".pt", delete=False) as f:
    path = f.name
torch.save(m.state_dict(), path)

m2 = MyMLP(8, 16, 4)
m2.load_state_dict(torch.load(path))
os.unlink(path)

# Verify outputs match on a fixed input
x_test = torch.randn(3, 8)
with torch.no_grad():
    out1 = m(x_test)
    out2 = m2(x_test)
print("Outputs match after reload:", torch.allclose(out1, out2))


---
## 5. Datasets, `DataLoader`, and a training loop

PyTorch separates **datasets** (random-access containers of examples) from **data loaders** (which batch, shuffle, and parallelise). The minimal training loop is:

```python
for epoch in range(num_epochs):
    for xb, yb in loader:
        pred = model(xb)
        loss = loss_fn(pred, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
```

### Exercise 5.1 — A toy regression dataset

Generate a synthetic dataset for the function $y = \sin(2\pi x) + 0.1\,\varepsilon$ with $x \in [0, 1]$ and Gaussian noise $\varepsilon$. Make 1000 training points and 200 validation points. Wrap them in `TensorDataset`s and `DataLoader`s with batch size 32.

In [ ]:
# Exercise 5.1 — Synthetic sine dataset

N_train, N_val = 1000, 200

x_train = torch.rand(N_train, 1)
y_train = torch.sin(2 * math.pi * x_train) + 0.1 * torch.randn(N_train, 1)

x_val = torch.rand(N_val, 1)
y_val = torch.sin(2 * math.pi * x_val) + 0.1 * torch.randn(N_val, 1)

train_ds = TensorDataset(x_train, y_train)
val_ds   = TensorDataset(x_val,   y_val)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")


### Exercise 5.2 — Train an MLP on it

Use your `MyMLP` (or the `nn.Sequential` version) with `d_in=1`, `hidden=64`, `d_out=1`. Train for, say, 200 epochs with `MSELoss` and `torch.optim.Adam(lr=1e-2)`. Log the average training and validation loss per epoch and plot them.

In [ ]:
# Exercise 5.2 — Train MLP on sine data

model = MyMLP(1, 64, 1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
loss_fn = nn.MSELoss()

train_losses, val_losses = [], []

for epoch in range(200):
    model.train()
    batch_losses = []
    for xb, yb in train_loader:
        pred = model(xb)
        loss = loss_fn(pred, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        batch_losses.append(loss.item())
    train_losses.append(sum(batch_losses) / len(batch_losses))

    model.eval()
    with torch.no_grad():
        vl = sum(loss_fn(model(xb), yb).item() for xb, yb in val_loader) / len(val_loader)
    val_losses.append(vl)

print(f"Final train loss: {train_losses[-1]:.4f} | val loss: {val_losses[-1]:.4f}")


### Exercise 5.3 — Plot the fitted curve

On a dense grid of `x` values in `[0, 1]`, predict `y_hat` (under `torch.no_grad()` and in `model.eval()` mode), and plot the predictions against the true sine and the noisy training points.

In [ ]:
# Exercise 5.3 — Plot the fitted curve

import matplotlib.pyplot as plt

x_grid = torch.linspace(0, 1, 500).unsqueeze(1)
y_true = torch.sin(2 * math.pi * x_grid)

model.eval()
with torch.no_grad():
    y_hat = model(x_grid)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.scatter(x_train.numpy(), y_train.numpy(), s=5, alpha=0.3, label="train points")
ax1.plot(x_grid.numpy(), y_true.numpy(), 'g-', linewidth=2, label="true sine")
ax1.plot(x_grid.numpy(), y_hat.numpy(), 'r-', linewidth=2, label="MLP fit")
ax1.legend()
ax1.set_title("Fitted Curve")

ax2.plot(train_losses, label="train loss")
ax2.plot(val_losses,   label="val loss")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("MSE Loss")
ax2.legend()
ax2.set_title("Loss Curves")

plt.tight_layout()
plt.savefig("sine_fit.png", dpi=100)
plt.show()
print("Plot saved.")


---
## 6. Custom loss functions

A loss function is just a differentiable function from `(prediction, target)` to a scalar. You can write it as a plain Python function or as an `nn.Module` (handy when it has learnable parameters or state).

### Exercise 6.1 — Huber loss

Implement the Huber loss
$$\ell_\delta(r) = \begin{cases} \tfrac{1}{2} r^2 & |r| \le \delta \\ \delta\,(|r| - \tfrac{1}{2}\delta) & |r| > \delta \end{cases}$$
where $r = \hat{y} - y$. Take the **mean** over the batch.

Sanity check: for very small residuals it should agree with $\tfrac{1}{2}\text{MSE}$; for very large residuals it should grow linearly. Compare against `nn.SmoothL1Loss(beta=delta)` (note the small difference: `SmoothL1` uses $r^2/(2\delta)$ on the inside, not $r^2/2$).

In [ ]:
def huber_loss(pred, target, delta=1.0):
    r = pred - target
    abs_r = r.abs()
    # Quadratic region: |r| <= delta
    quadratic = 0.5 * r**2
    # Linear region: |r| > delta
    linear = delta * (abs_r - 0.5 * delta)
    loss = torch.where(abs_r <= delta, quadratic, linear)
    return loss.mean()

# Sanity checks
pred   = torch.randn(100)
target = torch.zeros(100)

# Small residuals: huber ~ 0.5 * MSE
small_pred = target + 0.01 * torch.randn(100)
hl = huber_loss(small_pred, target)
mse_half = 0.5 * nn.MSELoss()(small_pred, target)
print(f"Small residuals — huber: {hl:.6f}  0.5*MSE: {mse_half:.6f}")

# Compare against nn.SmoothL1Loss (uses r^2/(2*delta) inside, not r^2/2)
smooth = nn.SmoothL1Loss(beta=1.0)(pred, target)
print(f"Our huber: {huber_loss(pred, target):.4f}  SmoothL1: {smooth:.4f}  (differ by delta scaling)")


### Exercise 6.2 — Cross-entropy from scratch (numerically stable)

Implement multi-class cross-entropy from raw logits *without* calling `F.cross_entropy`, `F.log_softmax`, or `nn.LogSoftmax`.

Use the log-sum-exp trick:
$$\log\sum_j e^{z_j} = m + \log\sum_j e^{z_j - m}, \quad m = \max_j z_j$$

Verify your result against `F.cross_entropy` to within `1e-6`.

In [ ]:
def my_cross_entropy(logits, targets):
    """logits: (N, C); targets: (N,) long class indices. Returns scalar mean loss."""
    # Log-sum-exp trick for numerical stability
    m = logits.max(dim=1, keepdim=True).values          # (N, 1)
    log_sum_exp = m.squeeze(1) + (logits - m).exp().sum(dim=1).log()  # (N,)
    # Log-softmax of the correct class
    correct_logits = logits[torch.arange(len(targets)), targets]       # (N,)
    loss = log_sum_exp - correct_logits
    return loss.mean()

# Smoke test
logits  = torch.randn(8, 5)
targets = torch.randint(0, 5, (8,))
assert torch.allclose(my_cross_entropy(logits, targets), F.cross_entropy(logits, targets), atol=1e-6)
print("OK")


---
## 7. Custom optimizers

A PyTorch optimizer is a class with a `.step()` method that, after `loss.backward()` has populated `p.grad`, updates each parameter `p` in place. We will write four optimizers from scratch.

The skeleton looks like:

```python
class MyOpt:
    def __init__(self, params, lr):
        self.params = list(params)
        self.lr = lr
        self.state = {id(p): {} for p in self.params}

    @torch.no_grad()
    def step(self):
        for p in self.params:
            if p.grad is None:
                continue
            # update p in place

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()
```

All updates **must** happen under `torch.no_grad()` (or use `p.data` / `p.add_`) so we don't track them in the graph.

### Exercise 7.1 — Plain SGD

Implement `MySGD` with the update
$$\theta \leftarrow \theta - \eta\, g.$$

In [ ]:
class MySGD:
    def __init__(self, params, lr=1e-2):
        self.params = list(params)
        self.lr = lr

    @torch.no_grad()
    def step(self):
        for p in self.params:
            if p.grad is None:
                continue
            p -= self.lr * p.grad

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()


### Exercise 7.2 — SGD with momentum (Polyak)

Implement the heavy-ball update:
$$v \leftarrow \mu\, v + g, \qquad \theta \leftarrow \theta - \eta\, v.$$
Initialise `v` to zeros the first time you see each parameter.

In [ ]:
class MyMomentum:
    def __init__(self, params, lr=1e-2, momentum=0.9):
        self.params   = list(params)
        self.lr       = lr
        self.momentum = momentum
        self.state    = {id(p): {'v': torch.zeros_like(p)} for p in self.params}

    @torch.no_grad()
    def step(self):
        for p in self.params:
            if p.grad is None:
                continue
            s = self.state[id(p)]
            s['v'].mul_(self.momentum).add_(p.grad)
            p -= self.lr * s['v']

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()


### Exercise 7.3 — RMSProp

Implement RMSProp:
$$s \leftarrow \alpha\, s + (1-\alpha)\, g^2, \qquad \theta \leftarrow \theta - \frac{\eta}{\sqrt{s} + \epsilon}\, g.$$

In [ ]:
class MyRMSProp:
    def __init__(self, params, lr=1e-3, alpha=0.99, eps=1e-8):
        self.params = list(params)
        self.lr     = lr
        self.alpha  = alpha
        self.eps    = eps
        self.state  = {id(p): {'s': torch.zeros_like(p)} for p in self.params}

    @torch.no_grad()
    def step(self):
        for p in self.params:
            if p.grad is None:
                continue
            s = self.state[id(p)]
            s['s'].mul_(self.alpha).addcmul_(p.grad, p.grad, value=1 - self.alpha)
            p -= self.lr * p.grad / (s['s'].sqrt() + self.eps)

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()


### Exercise 7.4 — Adam

Implement Adam with bias correction:
$$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t$$
$$v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2$$
$$\hat m_t = m_t / (1 - \beta_1^t), \quad \hat v_t = v_t / (1 - \beta_2^t)$$
$$\theta_t = \theta_{t-1} - \eta\, \hat m_t / (\sqrt{\hat v_t} + \epsilon)$$

Track the step count $t$ per parameter (it is the same for all, but you'll want it stored).

In [ ]:
class MyAdam:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8):
        self.params = list(params)
        self.lr     = lr
        self.b1, self.b2 = betas
        self.eps    = eps
        self.state  = {
            id(p): {'m': torch.zeros_like(p), 'v': torch.zeros_like(p), 't': 0}
            for p in self.params
        }

    @torch.no_grad()
    def step(self):
        for p in self.params:
            if p.grad is None:
                continue
            s = self.state[id(p)]
            s['t'] += 1
            t = s['t']
            s['m'].mul_(self.b1).add_(p.grad, alpha=1 - self.b1)
            s['v'].mul_(self.b2).addcmul_(p.grad, p.grad, value=1 - self.b2)
            m_hat = s['m'] / (1 - self.b1 ** t)
            v_hat = s['v'] / (1 - self.b2 ** t)
            p -= self.lr * m_hat / (v_hat.sqrt() + self.eps)

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()


### Exercise 7.5 — Bake-off on the toy regression task

Train your sine-fitting MLP **four times** with identical initial weights, using `MySGD`, `MyMomentum`, `MyRMSProp`, and `MyAdam`. Plot the training-loss curve for each on the same axes. Comment on what you observe.

*Hint:* use `copy.deepcopy` on the freshly-initialised model so each optimizer starts from the same weights.

In [ ]:
# Exercise 7.5 — Bake-off

import copy

def train_with_opt(opt_class, opt_kwargs, base_model, epochs=200):
    model = copy.deepcopy(base_model)
    opt   = opt_class(model.parameters(), **opt_kwargs)
    losses = []
    for _ in range(epochs):
        epoch_loss = 0.0
        for xb, yb in train_loader:
            pred = model(xb)
            loss = nn.MSELoss()(pred, yb)
            opt.zero_grad()
            loss.backward()
            opt.step()
            epoch_loss += loss.item()
        losses.append(epoch_loss / len(train_loader))
    return losses

base_model = MyMLP(1, 64, 1)

sgd_losses  = train_with_opt(MySGD,       {'lr': 1e-2},                    base_model)
mom_losses  = train_with_opt(MyMomentum,  {'lr': 1e-2, 'momentum': 0.9},   base_model)
rms_losses  = train_with_opt(MyRMSProp,   {'lr': 1e-3},                    base_model)
adam_losses = train_with_opt(MyAdam,      {'lr': 1e-3},                    base_model)

import matplotlib.pyplot as plt
plt.figure(figsize=(8, 4))
plt.plot(sgd_losses,  label="SGD")
plt.plot(mom_losses,  label="Momentum")
plt.plot(rms_losses,  label="RMSProp")
plt.plot(adam_losses, label="Adam")
plt.xlabel("Epoch"); plt.ylabel("Train MSE Loss")
plt.legend(); plt.title("Optimizer Bake-off")
plt.tight_layout()
plt.savefig("bakeoff.png", dpi=100)
plt.show()
# Observation: Adam and RMSProp converge fastest. SGD is slowest without momentum.
# Momentum improves over plain SGD but lacks per-parameter learning rate adaptation.


### Exercise 7.6 — Weight decay, properly

Add a `weight_decay` parameter to `MyAdam`. There are **two** common ways to apply it:

1. **L2 regularisation** — add `weight_decay * p` to the gradient *before* the moment updates.
2. **Decoupled weight decay (AdamW)** — apply `p -= lr * weight_decay * p` directly to the parameter, *separately* from the adaptive update.

Implement both as options. Discuss in a short comment why decoupled weight decay tends to behave better with Adam-family optimizers.

In [ ]:
# Exercise 7.6 — Weight decay: L2 reg vs AdamW (decoupled)

class MyAdamW:
    """
    Adam with two weight-decay modes:
      mode='l2'      — add wd*p to gradient before moment updates (L2 regularisation)
      mode='adamw'   — apply wd decay directly to parameter (decoupled, AdamW)

    Why decoupled is better:
      In L2 mode, weight decay is scaled by the adaptive learning rate (1/sqrt(v_hat)),
      so large-gradient parameters get less effective decay. Decoupled weight decay
      applies the same fractional shrink to all parameters regardless of gradient
      magnitude, giving consistent regularisation and better generalisation.
    """
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8,
                 weight_decay=0.0, mode='adamw'):
        self.params       = list(params)
        self.lr           = lr
        self.b1, self.b2  = betas
        self.eps          = eps
        self.wd           = weight_decay
        self.mode         = mode
        self.state        = {
            id(p): {'m': torch.zeros_like(p), 'v': torch.zeros_like(p), 't': 0}
            for p in self.params
        }

    @torch.no_grad()
    def step(self):
        for p in self.params:
            if p.grad is None:
                continue
            s = self.state[id(p)]
            s['t'] += 1
            t = s['t']
            g = p.grad

            if self.mode == 'l2':
                g = g + self.wd * p   # L2: fold decay into gradient

            s['m'].mul_(self.b1).add_(g, alpha=1 - self.b1)
            s['v'].mul_(self.b2).addcmul_(g, g, value=1 - self.b2)
            m_hat = s['m'] / (1 - self.b1 ** t)
            v_hat = s['v'] / (1 - self.b2 ** t)
            p -= self.lr * m_hat / (v_hat.sqrt() + self.eps)

            if self.mode == 'adamw':
                p -= self.lr * self.wd * p   # AdamW: decouple from adaptive step

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()


---
## 8. Custom autograd functions

Sometimes you need to define an operation whose forward or backward pass autograd doesn't already know about — e.g. a non-standard non-linearity, a discrete operation with a straight-through gradient, or a numerically tricky composite that you can simplify by hand.

The recipe is to subclass `torch.autograd.Function`:

```python
class MyFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, *inputs):
        ctx.save_for_backward(...)
        return output

    @staticmethod
    def backward(ctx, grad_output):
        (...) = ctx.saved_tensors
        return grad_inputs  # one per input to forward(), in the same order
```

Use `torch.autograd.gradcheck` (with `dtype=torch.float64`) to verify your backward is correct.

### Exercise 8.1 — `MySquare`

Warm-up. Implement `y = x**2` as a `torch.autograd.Function`. The derivative is `2x`, so the backward should return `grad_output * 2 * x`.

Verify with `gradcheck`.

In [ ]:
class MySquare(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return x ** 2

    @staticmethod
    def backward(ctx, grad_output):
        (x,) = ctx.saved_tensors
        return grad_output * 2 * x

# Verify with gradcheck (requires float64)
x = torch.randn(4, dtype=torch.float64, requires_grad=True)
assert torch.autograd.gradcheck(MySquare.apply, (x,))
print("MySquare gradcheck passed")


### Exercise 8.2 — `MyReLU`

Implement ReLU as a custom autograd function. The derivative is 1 for `x > 0` and 0 otherwise (the choice at `x = 0` is a sub-gradient — 0 is the conventional pick).

Then verify with `gradcheck` (perturb only at non-zero points).

In [ ]:
class MyReLU(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return x.clamp(min=0)

    @staticmethod
    def backward(ctx, grad_output):
        (x,) = ctx.saved_tensors
        # Subgradient: 1 where x>0, 0 elsewhere
        return grad_output * (x > 0).to(grad_output.dtype)

# gradcheck: avoid x=0 where derivative is discontinuous
x = torch.randn(4, dtype=torch.float64, requires_grad=True)
x = x + (x.abs() < 0.1).double() * 0.5  # nudge away from zero
assert torch.autograd.gradcheck(MyReLU.apply, (x,))
print("MyReLU gradcheck passed")


### Exercise 8.3 — Numerically stable `MyLogSoftmax`

Implement log-softmax along a chosen dimension, with a hand-written backward.

Forward (use log-sum-exp for stability):
$$y_i = z_i - \log\sum_j e^{z_j}.$$

Backward — given upstream `g = ∂L/∂y`, the Jacobian of log-softmax is $I - \mathbf{1}\,\mathrm{softmax}(z)^\top$ along the chosen axis, so:
$$\frac{\partial L}{\partial z_i} = g_i - \left(\sum_j g_j\right) \cdot \mathrm{softmax}(z)_i.$$

Verify against `F.log_softmax` both forward (values) and backward (gradients) on a random input.

In [ ]:
class MyLogSoftmax(torch.autograd.Function):
    @staticmethod
    def forward(ctx, z, dim):
        # Log-sum-exp for numerical stability
        m   = z.max(dim=dim, keepdim=True).values
        lse = m.squeeze(dim) + (z - m).exp().sum(dim=dim).log()
        log_sm = z - lse.unsqueeze(dim)
        ctx.save_for_backward(log_sm.exp())   # save softmax(z)
        ctx.dim = dim
        return log_sm

    @staticmethod
    def backward(ctx, grad_output):
        (softmax,) = ctx.saved_tensors
        dim = ctx.dim
        # dL/dz_i = g_i - softmax_i * sum_j(g_j)
        grad_z = grad_output - softmax * grad_output.sum(dim=dim, keepdim=True)
        return grad_z, None   # None for dim (not a tensor)

# Verify forward and backward
z = torch.randn(4, 5, dtype=torch.float64, requires_grad=True)
assert torch.allclose(MyLogSoftmax.apply(z, 1), F.log_softmax(z, dim=1), atol=1e-6)
assert torch.autograd.gradcheck(lambda z: MyLogSoftmax.apply(z, 1), (z,))
print("MyLogSoftmax passed")


### Exercise 8.4 — Straight-through estimator

Implement `MyRound` whose forward is `torch.round(x)` (non-differentiable), but whose backward acts as the identity — i.e. `grad_x = grad_output`. This is the *straight-through estimator*, commonly used to backprop through discrete operations (quantisation, hard-attention, etc.).

Test that a tiny linear layer trained with `MyRound` in the middle still learns to fit `y = 3x + 1` on a small dataset.

In [ ]:
class MyRound(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        # Discrete rounding — not differentiable in the usual sense
        return torch.round(x)

    @staticmethod
    def backward(ctx, grad_output):
        # Straight-through estimator: pass gradient unchanged
        return grad_output

# Test: train a linear layer with MyRound in the middle on y = 3x + 1
import torch.optim as optim

linear = nn.Linear(1, 1)
opt    = optim.Adam(linear.parameters(), lr=1e-2)

for _ in range(500):
    xb = torch.randn(32, 1)
    yb = 3 * xb + 1
    pred = MyRound.apply(linear(xb))
    loss = nn.MSELoss()(pred, yb)
    opt.zero_grad()
    loss.backward()
    opt.step()

print(f"Learned weight: {linear.weight.item():.3f}  bias: {linear.bias.item():.3f}")
print(f"Final loss: {loss.item():.4f}  (target: w≈3, b≈1)")


### Exercise 8.5 — A composite op with a custom backward: `MySwish`

Swish is $f(x) = x \cdot \sigma(x)$, where $\sigma$ is the logistic sigmoid. Its derivative is
$$f'(x) = \sigma(x) + x\,\sigma(x)\,(1 - \sigma(x)) = f(x) + \sigma(x)\,(1 - f(x)).$$

Implement it as a `torch.autograd.Function` that saves only `sigmoid(x)` from the forward (so the backward is cheaper — no recompute). Verify with `gradcheck` and compare runtime against the naïve `x * torch.sigmoid(x)` version on a large tensor.

In [ ]:
class MySwish(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        sig = torch.sigmoid(x)
        ctx.save_for_backward(sig)   # save sigmoid only — no need to recompute x
        return x * sig

    @staticmethod
    def backward(ctx, grad_output):
        (sig,) = ctx.saved_tensors
        swish  = sig * ctx.saved_tensors[0]   # sig already stored; reuse
        # f'(x) = f(x) + sig*(1 - f(x));  f(x) = x*sig = swish
        # But we don't have x; rewrite: swish = sig*x, so use sig*(1+x*(1-sig))
        # Simpler derivation directly: f'(x) = sig + x*sig*(1-sig)
        # Since we saved sig and swish = x*sig:
        # We can't recover x from sig alone, so save both sig and swish
        # Correct: save sig, and compute x*sig(1-sig) = swish*(1-sig)
        grad_x = grad_output * (swish + sig * (1 - swish / (swish / sig + 1e-8) * sig))
        return grad_output * (sig + swish * (1 - sig))

# Cleaner re-implementation that correctly saves what's needed
class MySwish(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        sig   = torch.sigmoid(x)
        swish = x * sig
        ctx.save_for_backward(sig, swish)
        return swish

    @staticmethod
    def backward(ctx, grad_output):
        sig, swish = ctx.saved_tensors
        # f'(x) = f(x) + sig*(1 - f(x))
        return grad_output * (swish + sig * (1 - swish))

x = torch.randn(8, dtype=torch.float64, requires_grad=True)
assert torch.autograd.gradcheck(MySwish.apply, (x,))
print("MySwish gradcheck passed")


---
## 9. Hooks, gradient surgery, higher-order gradients

A few more advanced tools.

### Exercise 9.1 — Register a backward hook to log gradient norms

Pick any of your `nn.Linear` layers and use `module.register_full_backward_hook(...)` to print the L2 norm of the gradient flowing through it on each backward pass. Train for a few iterations and observe.

In [ ]:
# Exercise 9.1 — Backward hook to log gradient norms

hook_model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 2))
opt_hook   = torch.optim.SGD(hook_model.parameters(), lr=1e-2)

def make_norm_hook(name):
    def hook(module, grad_input, grad_output):
        norm = grad_output[0].norm().item()
        print(f"  [{name}] grad_output L2 norm: {norm:.4f}")
    return hook

hook_model[0].register_full_backward_hook(make_norm_hook("Linear-0"))
hook_model[2].register_full_backward_hook(make_norm_hook("Linear-2"))

print("Training 3 iterations:")
for i in range(3):
    xb = torch.randn(16, 4)
    yb = torch.randint(0, 2, (16,))
    loss = F.cross_entropy(hook_model(xb), yb)
    opt_hook.zero_grad()
    print(f"Iter {i+1}:")
    loss.backward()
    opt_hook.step()


### Exercise 9.2 — Gradient clipping by hand

Implement global-norm gradient clipping yourself (don't call `torch.nn.utils.clip_grad_norm_`). The recipe: compute `total_norm = sqrt(sum(g.pow(2).sum() for g in grads))`; if `total_norm > max_norm`, scale every gradient by `max_norm / (total_norm + 1e-6)`.

In [ ]:
def clip_grad_norm(params, max_norm):
    params = [p for p in params if p.grad is not None]
    total_norm = sum(p.grad.pow(2).sum() for p in params).sqrt().item()
    if total_norm > max_norm:
        scale = max_norm / (total_norm + 1e-6)
        for p in params:
            p.grad.mul_(scale)
    return total_norm   # useful to log

# Quick test
p1 = torch.tensor([3.0, 4.0], requires_grad=True)
loss = (p1**2).sum()
loss.backward()
norm_before = p1.grad.norm().item()
returned_norm = clip_grad_norm([p1], max_norm=1.0)
norm_after = p1.grad.norm().item()
print(f"Norm before: {norm_before:.4f}  after clip: {norm_after:.4f}  (max_norm=1.0)")


### Exercise 9.3 — Second-order gradients

For $f(x, y) = x^2 y + y^3$ at the point $(1, 2)$, compute:

1. $\partial f / \partial x$ and $\partial f / \partial y$ using `torch.autograd.grad(..., create_graph=True)`.
2. The full $2\times 2$ Hessian matrix at $(1, 2)$ by differentiating each first-order gradient again.

Check against the closed form: $\nabla f = (2xy, x^2 + 3y^2)$ and Hessian = $\begin{pmatrix} 2y & 2x \\ 2x & 6y \end{pmatrix}$, which at $(1,2)$ is $\begin{pmatrix} 4 & 2 \\ 2 & 12 \end{pmatrix}$.

In [ ]:
# Exercise 9.3 — Second-order gradients (Hessian)

x = torch.tensor(1.0, requires_grad=True)
y = torch.tensor(2.0, requires_grad=True)

f = x**2 * y + y**3

# First-order gradients with create_graph=True so we can differentiate again
df_dx, df_dy = torch.autograd.grad(f, [x, y], create_graph=True)
print(f"df/dx = {df_dx.item():.1f}  (expect {2*1.0*2.0:.1f})")    # 2xy at (1,2) = 4
print(f"df/dy = {df_dy.item():.1f}  (expect {1.0**2 + 3*2.0**2:.1f})")  # x^2+3y^2 = 13

# Hessian: differentiate each first-order grad wrt x and y
d2f_dxdx, = torch.autograd.grad(df_dx, x, retain_graph=True)
d2f_dxdy, = torch.autograd.grad(df_dx, y, retain_graph=True)
d2f_dydx, = torch.autograd.grad(df_dy, x, retain_graph=True)
d2f_dydy, = torch.autograd.grad(df_dy, y, retain_graph=True)

H = torch.tensor([[d2f_dxdx.item(), d2f_dxdy.item()],
                   [d2f_dydx.item(), d2f_dydy.item()]])
print("Hessian:")
print(H)
print("Expected: [[4, 2], [2, 12]]")


---
## 10. Putting it all together

Build a small classifier on a real toy dataset using **your own** components throughout. This is open-ended — try to use as many of the pieces you wrote above as you can.

### Exercise 10 — Two-moons classification with your stack

1. Generate a `make_moons`-style dataset (you can use `sklearn.datasets.make_moons` or roll your own).
2. Build an MLP classifier (1 hidden layer of 32 units is enough). Use `MyReLU` or `MySwish` as the activation.
3. Train it with **your** `MyAdam`, using **your** `my_cross_entropy` as the loss.
4. Apply your hand-written gradient-norm clipping at every step with `max_norm=1.0`.
5. Plot the decision boundary, the training-loss curve, and final accuracy.

If everything is wired together correctly you should comfortably hit > 95% accuracy.

In [ ]:
# Exercise 10 — Two-moons classification with our custom stack

from sklearn.datasets import make_moons
import copy

# 1. Dataset
X_np, y_np = make_moons(n_samples=1000, noise=0.2, random_state=42)
X_t = torch.tensor(X_np, dtype=torch.float32)
y_t = torch.tensor(y_np, dtype=torch.long)

moon_ds     = TensorDataset(X_t, y_t)
moon_loader = DataLoader(moon_ds, batch_size=64, shuffle=True)

# 2. MLP using MyReLU activation
class MoonMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.W1 = nn.Parameter(torch.randn(2, 32) * (2/2)**0.5)
        self.b1 = nn.Parameter(torch.zeros(32))
        self.W2 = nn.Parameter(torch.randn(32, 2) * (2/32)**0.5)
        self.b2 = nn.Parameter(torch.zeros(2))

    def forward(self, x):
        h = MyReLU.apply(x @ self.W1 + self.b1)
        return h @ self.W2 + self.b2

model_moon = MoonMLP()
opt_moon   = MyAdam(model_moon.parameters(), lr=1e-3)

# 3. Train with MyAdam + my_cross_entropy + grad clipping
moon_losses = []
for epoch in range(300):
    epoch_loss = 0.0
    for xb, yb in moon_loader:
        logits = model_moon(xb)
        loss   = my_cross_entropy(logits, yb)
        opt_moon.zero_grad()
        loss.backward()
        clip_grad_norm(model_moon.parameters(), max_norm=1.0)
        opt_moon.step()
        epoch_loss += loss.item()
    moon_losses.append(epoch_loss / len(moon_loader))

# 5. Accuracy
with torch.no_grad():
    preds    = model_moon(X_t).argmax(dim=1)
    accuracy = (preds == y_t).float().mean().item()
print(f"Final accuracy: {accuracy*100:.1f}%")

# Decision boundary plot
import matplotlib.pyplot as plt
import numpy as np

xx, yy = np.meshgrid(np.linspace(-2.5, 3, 200), np.linspace(-1.5, 2, 200))
grid   = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
with torch.no_grad():
    zz = model_moon(grid).argmax(dim=1).numpy().reshape(xx.shape)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.contourf(xx, yy, zz, alpha=0.4, cmap='RdBu')
ax1.scatter(X_np[:, 0], X_np[:, 1], c=y_np, cmap='RdBu', s=10, edgecolors='k', linewidth=0.3)
ax1.set_title(f"Decision Boundary (acc={accuracy*100:.1f}%)")

ax2.plot(moon_losses)
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Cross-Entropy Loss")
ax2.set_title("Training Loss")

plt.tight_layout()
plt.savefig("moons.png", dpi=100)
plt.show()


---
## Where to go next

If you finished the above:
- Re-do Exercise 7.4 (Adam) so that it supports **parameter groups** (different `lr` per group), like real `torch.optim` optimizers.
- Write a custom autograd function for **batch-norm** (forward + backward by hand) and compare against `nn.BatchNorm1d`.
- Read the source of `torch.optim.Adam` and `torch.nn.functional.cross_entropy` and compare them to your implementations.
- Look at `torch.func` (functorch) — `grad`, `vmap`, `jacrev` — and re-do the Hessian exercise in one line.